## task 1. plagiarism detection performance (binary)

In [ ]:
import pandas as pd
df = pd.read_csv("plagbench_evaluation_set.csv")
df

In [ ]:
import requests

API_URL = "https://api-inference.huggingface.co/models/meta-llama/Llama-2-70b-chat-hf"
headers = {"Authorization": YOUR API}

def query(payload):
	response = requests.post(API_URL, headers=headers, json=payload)
	return response.json()
	
output = query({
	"inputs": "HELLO",
    "parameters": {"max_new_tokens":50, "do_sample" : False}
})

In [ ]:
output = query({
	"inputs": "test",
    "parameters": {"max_new_tokens":50, "do_sample" : False}
})

print(output)

In [ ]:
!pip install openai==0.28

In [ ]:
import requests
import pandas as pd
import json
import os
import openai
import time

api_key = YOUR API # API key
openai.api_key = api_key

def gpt3_query(prompt):
    #vanilla_prompt = f'''Does text A contain verbatim copies (longer than 50 character length) from text B? (yes/no).
    LLM_genrated_text = openai.ChatCompletion.create(
            #model="gpt-3.5-turbo-16k", #"text-davinci-003",
            model = "gpt-3.5-turbo-1106",
            max_tokens=400,
            temperature=0.0,
            messages=[
                {"role": "user", "content": prompt},
              ],
    )
    return LLM_genrated_text['choices'][0]['message']['content']



def gpt4_query(prompt):
    #vanilla_prompt = f'''Does text A contain verbatim copies (longer than 50 character length) from text B? (yes/no).
    LLM_genrated_text = openai.ChatCompletion.create(
            #model="gpt-3.5-turbo-16k", #"text-davinci-003",
            model = "gpt-4-1106-preview",
            max_tokens=400,
            temperature=0.0,
            messages=[
                {"role": "user", "content": prompt},
              ],
    )
    return LLM_genrated_text['choices'][0]['message']['content']



def llama_query(payload):
    API_URL = "https://api-inference.huggingface.co/models/meta-llama/Llama-2-70b-chat-hf"
    headers = {"Authorization":  YOUR API}  
    response = requests.post(API_URL, headers=headers, json=payload)
    print(response)
    return response.json()


def llama3_query(payload):
    API_URL = "https://api-inference.huggingface.co/models/meta-llama/Meta-Llama-3-70B-Instruct"
    headers = {"Authorization":  YOUR API}  
    response = requests.post(API_URL, headers=headers, json=payload)
    print(response)
    return response.json()

def mixtral_query(payload):
    API_URL = "https://api-inference.huggingface.co/models/mistralai/Mixtral-8x7B-Instruct-v0.1"
    headers = {"Authorization":  YOUR API}  
    response = requests.post(API_URL, headers=headers, json=payload)
    print(response)
    return response.json()

   
def evaluate_gpt_metrics(source_doc, susp_doc):
    vanilla_prompt = f'''There are three types of plagiarism:
• Verbatim plagiarism: the evaluation text consists of exact copies of words or phrases without transformation from the source text without citation.
• Paraphrase plagiarism: the evaluation text is rephrased or rewritten using different words but retain the same meaning and structure as the source text without citation.
• Summary plagiarism: the evaluation text encapsulates the most essential points of the source text into a shorter form without citation.

If the evaluation text does not belong to any of three plagiarism categories, it means plagiarism-free.
Given a pair of text and provided plagiarism definitions, does the evaluation text plagiarize the source text (yes/no)? Please format your final response as ‘Answer: {{response}}’.  

Source text: {source_doc}
Evaluation text: {susp_doc}
'''

    prompt = f'''There are three types of plagiarism:
• Verbatim plagiarism: the evaluation text consists of exact copies of words or phrases without transformation from the source text without citation.
• Paraphrase plagiarism: the evaluation text is rephrased or rewritten using different words but retain the same meaning and structure as the source text without citation.
• Summary plagiarism: the evaluation text encapsulates the most essential points of the source text into a shorter form without citation.

If the evaluation text does not belong to any of three plagiarism categories, it means plagiarism-free.
Given a pair of text and provided plagiarism definitions, does the evaluation text plagiarize the source text (yes/no)? First think step-by-step and format your final response as ‘Final answer: {{response}}’.  

Source text: {source_doc}
Evaluation text: {susp_doc}

Answer: Let's think step-by-step.'''
    
    gpt3_response = gpt3_query(prompt)
    gpt4_response = gpt4_query(prompt)

    return {"gpt3": gpt3_response, "gpt4":gpt4_response}
    
    


def evaluate_metrics(source_doc, susp_doc):
    vanilla_prompt = f'''There are three types of plagiarism:
• Verbatim plagiarism: the evaluation text consists of exact copies of words or phrases without transformation from the source text without citation.
• Paraphrase plagiarism: the evaluation text is rephrased or rewritten using different words but retain the same meaning and structure as the source text without citation.
• Summary plagiarism: the evaluation text encapsulates the most essential points of the source text into a shorter form without citation.

If the evaluation text does not belong to any of three plagiarism categories, it means plagiarism-free.
Given a pair of text and provided plagiarism definitions, does the evaluation text plagiarize the source text (yes/no)? Please format your final response as ‘Answer: {{response}}’.  

Source text: {source_doc}
Evaluation text: {susp_doc}
'''

    prompt = f'''There are three types of plagiarism:
• Verbatim plagiarism: the evaluation text consists of exact copies of words or phrases without transformation from the source text without citation.
• Paraphrase plagiarism: the evaluation text is rephrased or rewritten using different words but retain the same meaning and structure as the source text without citation.
• Summary plagiarism: the evaluation text encapsulates the most essential points of the source text into a shorter form without citation.

If the evaluation text does not belong to any of three plagiarism categories, it means plagiarism-free.
Given a pair of text and provided plagiarism definitions, does the evaluation text plagiarize the source text (yes/no)? First think step-by-step and format your final response as ‘Final answer: {{response}}’.  

Source text: {source_doc}
Evaluation text: {susp_doc}

Answer: Let's think step-by-step.'''


    llama_response = llama_query({"inputs": vanilla_prompt,  "parameters": {"max_new_tokens":400, "do_sample" : False}})
    mixtral_response = mixtral_query({"inputs": prompt, "parameters": {"max_new_tokens":400, "do_sample" : False}})
    llama3_response = llama3_query({"inputs": prompt, "parameters": {"max_new_tokens":400, "do_sample" : False}})

    
    
    # print(response)
    response2 = llama_response[0].get("generated_text", "").replace(vanilla_prompt,"") 
    response3 = mixtral_response[0].get("generated_text", "").replace(prompt,"") 
    response4 = llama3_response[0].get("generated_text", "").replace(prompt,"")

    return {"llama": response2, "mistral":response3, "llama3": response4}

existing_data = []
correct_llama = 0
correct_mistral = 0

final_df =df 

# Load the CSV file
for index, row in final_df.iterrows():
    print(index)
    original_text = row["source_doc"]
    paraphrase_text = row["susp_doc"]

    output = evaluate_metrics(original_text, paraphrase_text)
    output2 = evaluate_gpt_metrics(original_text, paraphrase_text)

    #print(output)
    #print(output2)

    output2.update(output)

    #correct_llama += f'''answer: {row['label']}''' in output['llama'].lower()
    #correct_mistral += f'''answer: {row['label']}''' in output['mistral'].lower()
    #print(correct_llama, correct_mistral)
   

    # write files
    loc = '.'
    filename = f'binary_detection_mistral_llama3_gpts_vanilla_cot_new.json'

    # make dir
    os.makedirs(f"{loc}", exist_ok=True)

    try:
        with open(f'{loc}/{filename}', 'r') as file:
            existing_data = json.load(file)
    except FileNotFoundError:
        # If the file doesn't exist, create an empty list as the starting point
        existing_data = []

    # Step 3: Modify the data structure by adding the new item
    #new_item = {"susp_doc": df['susp_doc'][i], "source_doc": df['source_doc'][i], "prediction": data[0]['generated_text'], "label": df['label'][i], "genre": df['genre'][i]}  # Replace with your item
    existing_data.append(output2)
    
    # Step 4: Write the updated data structure back to the JSON file
    with open(f'{loc}/{filename}', 'w') as file:
        json.dump(existing_data, file, indent=4)


### task 2.

In [ ]:
import pandas as pd
final_df = pd.read_csv("plagbench_evaluation_set.csv")
final_df

In [ ]:
import requests
import pandas as pd
import json
import os
import openai
import time

api_key = YOUR KEY # API key
openai.api_key = YOUR KEY

def gpt3_query(prompt):
    #vanilla_prompt = f'''Does text A contain verbatim copies (longer than 50 character length) from text B? (yes/no).
    LLM_genrated_text = openai.ChatCompletion.create(
            #model="gpt-3.5-turbo-16k", #"text-davinci-003",
            model = "gpt-3.5-turbo-1106",
            max_tokens=400,
            temperature=0.0,
            messages=[
                {"role": "user", "content": prompt},
              ],
    )
    return LLM_genrated_text['choices'][0]['message']['content']



def gpt4_query(prompt):
    #vanilla_prompt = f'''Does text A contain verbatim copies (longer than 50 character length) from text B? (yes/no).
    LLM_genrated_text = openai.ChatCompletion.create(
            #model="gpt-3.5-turbo-16k", #"text-davinci-003",
            model = "gpt-4-1106-preview",
            max_tokens=400,
            temperature=0.0,
            messages=[
                {"role": "user", "content": prompt},
              ],
    )
    return LLM_genrated_text['choices'][0]['message']['content']


def llama_query(payload):
    API_URL = "https://api-inference.huggingface.co/models/meta-llama/Llama-2-70b-chat-hf"
    headers = {"Authorization": YOUR API}  
    response = requests.post(API_URL, headers=headers, json=payload)
    print(response)
    return response.json()


def llama3_query(payload):
    API_URL = "https://api-inference.huggingface.co/models/meta-llama/Meta-Llama-3-70B-Instruct"
    headers = {"Authorization": YOUR API}  
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

def mixtral_query(payload):
    API_URL = "https://api-inference.huggingface.co/models/mistralai/Mixtral-8x7B-Instruct-v0.1"
    headers = {"Authorization": YOUR API}  
    response = requests.post(API_URL, headers=headers, json=payload)
    print(response)
    return response.json()

   
def evaluate_gpt_metrics(source_doc, susp_doc):
    vanilla_prompt = f'''There are three types of plagiarism:
• Verbatim plagiarism: the evaluation text consists of exact copies of words or phrases without transformation from the source text without citation.
• Paraphrase plagiarism: the evaluation text is rephrased or rewritten using different words but retain the same meaning and structure as the source text without citation.
• Summary plagiarism: the evaluation text encapsulates the most essential points of the source text into a shorter form without citation.
• No plagiarism: the evaluation text does not belong to any of three plagiarism categories.

Given a pair of text and provided plagiarism definitions, what type of plagiarism (no/verbatim/paraphrase/summary) does the evaluation belong to when compared to the source text? Please format your final response as ‘Answer: {{response}}’.  

Source text: {source_doc}
Evaluation text: {susp_doc}
'''

    prompt = f'''There are three types of plagiarism:
• Verbatim plagiarism: the evaluation text consists of exact copies of words or phrases without transformation from the source text without citation.
• Paraphrase plagiarism: the evaluation text is rephrased or rewritten using different words but retain the same meaning and structure as the source text without citation.
• Summary plagiarism: the evaluation text encapsulates the most essential points of the source text into a shorter form without citation.
• No plagiarism: the evaluation text does not belong to any of three plagiarism categories.

Given a pair of text and provided plagiarism definitions, what type of plagiarism (no/verbatim/paraphrase/summary) does the evaluation belong to when compared to the source text? First think step-by-step and format your final response as ‘Final Answer: {{response}}’.  

Source text: {source_doc}
Evaluation text: {susp_doc}

Answer: Let's think step-by-step.'''
    
    gpt3_response = gpt3_query(prompt)
    gpt4_response = gpt4_query(prompt)

    return {"gpt3": gpt3_response, "gpt4":gpt4_response}
    
    


def evaluate_metrics(source_doc, susp_doc):
    vanilla_prompt = f'''There are three types of plagiarism:
• Verbatim plagiarism: the evaluation text consists of exact copies of words or phrases without transformation from the source text without citation.
• Paraphrase plagiarism: the evaluation text is rephrased or rewritten using different words but retain the same meaning and structure as the source text without citation.
• Summary plagiarism: the evaluation text encapsulates the most essential points of the source text into a shorter form without citation.
• No plagiarism: the evaluation text does not belong to any of three plagiarism categories.

Given a pair of text and provided plagiarism definitions, what type of plagiarism (no/verbatim/paraphrase/summary) does the evaluation belong to when compared to the source text? Please format your final response as ‘Answer: {{response}}’.  

Source text: {source_doc}
Evaluation text: {susp_doc}
'''

    prompt = f'''There are three types of plagiarism:
• Verbatim plagiarism: the evaluation text consists of exact copies of words or phrases without transformation from the source text without citation.
• Paraphrase plagiarism: the evaluation text is rephrased or rewritten using different words but retain the same meaning and structure as the source text without citation.
• Summary plagiarism: the evaluation text encapsulates the most essential points of the source text into a shorter form without citation.
• No plagiarism: the evaluation text does not belong to any of three plagiarism categories.

Given a pair of text and provided plagiarism definitions, what type of plagiarism (no/verbatim/paraphrase/summary) does the evaluation belong to when compared to the source text? First think step-by-step and format your final response as ‘Final Answer: {{response}}’.  

Source text: {source_doc}
Evaluation text: {susp_doc}

Answer: Let's think step-by-step.'''


    #gemma_response = gemma_query({"inputs": prompt})
    #time.sleep(2.0)
    
    llama3_response = llama3_query({"inputs": prompt, "parameters": {"max_new_tokens":400, "do_sample" : False}})
    llama_response = llama_query({"inputs": vanilla_prompt,  "parameters": {"max_new_tokens":50, "do_sample" : False}})
    #time.sleep(1.0)
    mixtral_response = mixtral_query({"inputs": prompt, "parameters": {"max_new_tokens":400, "do_sample" : False}})


    
    
    # print(response)
    response2 = llama_response[0].get("generated_text", "").replace(prompt,"") 
    response3 = mixtral_response[0].get("generated_text", "").replace(prompt,"") 
    response4 = llama3_response[0].get("generated_text", "").replace(prompt,"")
    #response4 = phi3_response[0].get("generated_text", "").replace(prompt,"") if phi3_response and isinstance(phi3_response, list) and phi3_response[0] else ""    

    #print(response1)    
    #print(response2)    
    #print(response3)    
    #print(response4)  

    #results1 = get_response(response1)
    #results2 = get_response(response2)
    #results3 = get_response(response3)
    #results4 = get_response(response4)

    #print(results1)    
    #print(results2)    
    #print(results3)    
    #print(results4)  
    
    return {"llama2":response2, "mixtral": response3, "llama3": response4}
    #return {"llama3": response4}

existing_data = []
correct_llama = 0
correct_mistral = 0

# Load the CSV file
for index, row in final_df.iterrows():
    print(index)
    original_text = row["source_doc"]
    paraphrase_text = row["susp_doc"]

    output = evaluate_metrics(original_text, paraphrase_text)
    output2 = evaluate_gpt_metrics(original_text, paraphrase_text)

    #print(output)
    #print(output2)

    output2.update(output)
    #output2.update({"gold_label": row['plagiarism_type']})

    #correct_llama += f'''answer: {row['label']}''' in output['llama'].lower()
    #correct_mistral += f'''answer: {row['label']}''' in output['mistral'].lower()
    #print(correct_llama, correct_mistral)
   

    # write files
    loc = '.'
    filename = f'category_detection_mistral_llama3_gpts_zheoshot_cot_new.json'

    # make dir
    os.makedirs(f"{loc}", exist_ok=True)

    try:
        with open(f'{loc}/{filename}', 'r') as file:
            existing_data = json.load(file)
    except FileNotFoundError:
        # If the file doesn't exist, create an empty list as the starting point
        existing_data = []

    # Step 3: Modify the data structure by adding the new item
    #new_item = {"susp_doc": df['susp_doc'][i], "source_doc": df['source_doc'][i], "prediction": data[0]['generated_text'], "label": df['label'][i], "genre": df['genre'][i]}  # Replace with your item
    existing_data.append(output)
    
    # Step 4: Write the updated data structure back to the JSON file
    with open(f'{loc}/{filename}', 'w') as file:
        json.dump(existing_data, file, indent=4)
